<a href="https://colab.research.google.com/github/erymuchyarh-design/onyx/blob/main/RF_DETR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Drive Setting

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Access Google Drive Folder
import os
os.chdir("gdrive/MyDrive")

# Installation

In [ ]:
# Create RFDETR root folder
!mkdir rfdetr

In [ ]:
# Go to RFDETR root folder
%cd rfdetr

In [ ]:
# Install RFDETR
%pip install rfdetr

# Detection

## Download Resources

In [ ]:
!gdown https://drive.google.com/uc?id=16t4AFWoUtGLoKaUHIBFWD2kXy1HOSEOH
!unzip inference.zip

## Detect Image

In [ ]:
import cv2
import supervision as sv
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
from rfdetr.assets.coco_classes import COCO_CLASSES

def detect_image(variant, image, threshold=0.5):
    if variant == "nano":
        model = RFDETRNano()
    elif variant == "small":
        model = RFDETRSmall()
    elif variant == "medium":
        model = RFDETRMedium()
    elif variant == "large":
        model = RFDETRLarge()
    else:
        raise ValueError(f"Unknown model variant: {args.variant}")

    detections = model.predict(image, threshold=threshold)

    labels = [f"{COCO_CLASSES[class_id]}" for class_id in detections.class_id]

    annotated_image = sv.BoxAnnotator().annotate(detections.metadata["source_image"], detections)
    annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

    cv2.imwrite("output_image.jpg", annotated_image)
    cv2.destroyAllWindows()

In [ ]:
detect_image('nano', 'inference/drone.jpg')

In [ ]:
from google.colab import files

files.download("output_image.jpg")

## Detect Video

In [ ]:
import cv2
import supervision as sv
import time
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
from rfdetr.assets.coco_classes import COCO_CLASSES

# Function to display FPS (Frames Per Second) on the frame
def show_fps(frame, fps, avg_fps):
    # ----- Siapkan teks -----
    lines = [
        f"FPS: {fps:.2f}",
        f"AVG FPS: {avg_fps:.2f}"
    ]

    font = cv2.FONT_HERSHEY_PLAIN
    font_scale = 5
    thickness = 5

    # Hitung ukuran teks terbesar
    max_width = 0
    total_height = 0
    line_heights = []

    for line in lines:
        (text_w, text_h), _ = cv2.getTextSize(line, font, font_scale, thickness)
        max_width = max(max_width, text_w)
        line_heights.append(text_h)
        total_height += text_h + 10  # padding antar baris

    # ----- Tentukan posisi kotak -----
    x, y = 10, 10
    padding = 20

    box_w = max_width + padding * 2
    box_h = total_height + padding

    # ----- Gambar kotak transparan -----
    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + box_w, y + box_h), (0, 0, 0), -1)
    alpha = 0.6  # tingkat transparansi

    # Gabungkan frame dengan overlay transparan
    frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    # ----- Tulis teks ke frame -----
    y_offset = y + padding + line_heights[0]

    for i, line in enumerate(lines):
        cv2.putText(
            frame,
            line,
            (x + padding, y_offset),
            font,
            font_scale,
            (0, 255, 0),
            thickness
        )
        if i < len(lines) - 1:
            y_offset += line_heights[i+1] + 10

    return frame

def detect_video(variant, video, threshold=0.5):
    if variant == "nano":
        model = RFDETRNano()
    elif variant == "small":
        model = RFDETRSmall()
    elif variant == "medium":
        model = RFDETRMedium()
    elif variant == "large":
        model = RFDETRLarge()
    else:
        raise ValueError(f"Unknown model variant: {args.variant}")

    video_capture = cv2.VideoCapture(video)  # Open the video source (file or webcam)
    if not video_capture.isOpened():
        raise RuntimeError("Failed to open video source: <SOURCE_VIDEO_PATH>")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = video_capture.get(cv2.CAP_PROP_FPS)
    width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_video = cv2.VideoWriter(video.replace(".mp4", "_output.mp4"), fourcc, fps, (width, height))
    total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_count = 0
    total_fps = 0
    avg_fps = 0

    while True:
        success, frame_bgr = video_capture.read()
        if not success:
            break

        start_time = time.time()
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        detections = model.predict(frame_rgb, threshold=threshold)

        end_time = time.time()

        labels = [COCO_CLASSES[class_id] for class_id in detections.class_id]

        annotated_frame = sv.BoxAnnotator().annotate(frame_bgr, detections)
        annotated_frame = sv.LabelAnnotator().annotate(annotated_frame, detections, labels)

        fps = 1 / (end_time - start_time)
        total_fps += fps
        frame_count += 1
        avg_fps = total_fps / frame_count

        annotated_frame = show_fps(annotated_frame, fps, avg_fps)

        print("Frame Processed", frame_count, " : ", total_frames)

        output_video.write(annotated_frame)

    video_capture.release()
    output_video.release()
    cv2.destroyAllWindows()

In [ ]:
detect_video('nano', 'inference/city_road.mp4')

In [ ]:
from google.colab import files

files.download("inference/city_road_output.mp4")

# Training

In [ ]:
# Install Logger
!pip install "rfdetr[train,loggers]"

In [ ]:
# Re-install OpenCV Python
!pip uninstall opencv-python opencv-python-headless -y
!pip install opencv-python

## Download Dataset

In [ ]:
!mkdir datasets

In [ ]:
!gdown https://drive.google.com/uc?id=1_rDCF-c1dizim-CxJZHMkFx5Di6v0qUm

In [ ]:
!unzip road_sign_dataset.zip -d datasets

## Training Process

In [ ]:
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge

def train_model(variant, dataset_dir, epochs, output_dir, device='cuda', batch_size=2, grad_accum_steps=8):
    if variant == "nano":
        model = RFDETRNano()
    elif variant == "small":
        model = RFDETRSmall()
    elif variant == "medium":
        model = RFDETRMedium()
    elif variant == "large":
        model = RFDETRLarge()
    else:
        raise ValueError(f"Unknown model variant: {variant}")

    model.train(
        dataset_dir=dataset_dir,
        epochs=epochs,
        device=device,
        batch_size=batch_size,
        grad_accum_steps=grad_accum_steps,
        lr=1e-4,
        output_dir=output_dir,
    )

In [ ]:
train_model('nano', 'datasets/road_sign_dataset/', 100, 'rfdetr_road_sign', device='cuda', batch_size=4, grad_accum_steps=8)

## Detect Image

In [ ]:
import cv2
import supervision as sv
from rfdetr import RFDETR
from rfdetr.assets.coco_classes import COCO_CLASSES

CLASSES = ['Speedlimit', 'Trafficlight', 'Stop', 'Crosswalk']

def detect_image_custom(model, image, threshold=0.5):
    model = RFDETR.from_checkpoint(model)

    detections = model.predict(image, threshold=threshold)

    labels = [f"{CLASSES[class_id]}" for class_id in detections.class_id]

    annotated_image = sv.BoxAnnotator().annotate(detections.metadata["source_image"], detections)
    annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

    annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

    cv2.imwrite("output_image_custom.jpg", annotated_image)
    cv2.destroyAllWindows()

In [ ]:
detect_image_custom('rfdetr_road_sign/checkpoint_best_regular.pth', 'inference/road-sign.jpg')

In [ ]:
from google.colab import files

files.download("output_image_custom.jpg")

## Detect Video

In [ ]:
import cv2
import supervision as sv
import argparse
import time
from rfdetr import RFDETR

CLASSES = ['Speedlimit', 'Trafficlight', 'Stop', 'Crosswalk']

# Function to display FPS (Frames Per Second) on the frame
def show_fps(frame, fps, avg_fps):
    # ----- Siapkan teks -----
    lines = [
        f"FPS: {fps:.2f}",
        f"AVG FPS: {avg_fps:.2f}"
    ]

    font = cv2.FONT_HERSHEY_PLAIN
    font_scale = 5
    thickness = 5

    # Hitung ukuran teks terbesar
    max_width = 0
    total_height = 0
    line_heights = []

    for line in lines:
        (text_w, text_h), _ = cv2.getTextSize(line, font, font_scale, thickness)
        max_width = max(max_width, text_w)
        line_heights.append(text_h)
        total_height += text_h + 10  # padding antar baris

    # ----- Tentukan posisi kotak -----
    x, y = 10, 10
    padding = 20

    box_w = max_width + padding * 2
    box_h = total_height + padding

    # ----- Gambar kotak transparan -----
    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + box_w, y + box_h), (0, 0, 0), -1)
    alpha = 0.6  # tingkat transparansi

    # Gabungkan frame dengan overlay transparan
    frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    # ----- Tulis teks ke frame -----
    y_offset = y + padding + line_heights[0]

    for i, line in enumerate(lines):
        cv2.putText(
            frame,
            line,
            (x + padding, y_offset),
            font,
            font_scale,
            (0, 255, 0),
            thickness
        )
        if i < len(lines) - 1:
            y_offset += line_heights[i+1] + 10

    return frame

def detect_video_custom(model, video, threshold=0.5):

    model = RFDETR.from_checkpoint(model)  # Load the model from the specified path

    video_capture = cv2.VideoCapture(video)  # Open the video source (file or webcam)
    if not video_capture.isOpened():
        raise RuntimeError("Failed to open video source: <SOURCE_VIDEO_PATH>")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = video_capture.get(cv2.CAP_PROP_FPS)
    width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    output_video = cv2.VideoWriter(video.replace(".mp4", "_output.mp4"), fourcc, fps, (width, height))
    total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_count = 0
    total_fps = 0
    avg_fps = 0

    while True:
        success, frame_bgr = video_capture.read()
        if not success:
            break

        start_time = time.time()
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        detections = model.predict(frame_rgb, threshold=threshold)

        end_time = time.time()

        labels = [CLASSES[class_id] for class_id in detections.class_id]

        annotated_frame = sv.BoxAnnotator().annotate(frame_bgr, detections)
        annotated_frame = sv.LabelAnnotator().annotate(annotated_frame, detections, labels)

        fps = 1 / (end_time - start_time)
        total_fps += fps
        frame_count += 1
        avg_fps = total_fps / frame_count

        annotated_frame = show_fps(annotated_frame, fps, avg_fps)
        output_video.write(annotated_frame)

        print("Frame Processed", frame_count, " : ", total_frames)

    video_capture.release()
    output_video.release()
    cv2.destroyAllWindows()

In [ ]:
detect_image_custom('rfdetr_road_sign/checkpoint_best_regular.pth', 'inference/road-sign.mp4')

# Vehicle Counter

## Install Trackers

In [ ]:
!pip install trackers

## Download the Model

In [ ]:
# Download the model
!gdown https://drive.google.com/uc?id=1wPJTwg1GtV05sO-vgpShIzuip-oz28nh

## Install ONNX

In [ ]:
# Install onnx onnxruntime-gpu
!pip install onnx onnxruntime-gpu

## Run the program

In [ ]:
import numpy as np
import time
from datetime import datetime

import cv2
import onnxruntime as ort

import supervision as sv
from rfdetr.assets.coco_classes import COCO_CLASSES
import json
from datetime import datetime

from trackers import BoTSORTTracker

INPUT_SIZE = 384
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Data
counts = {
    "total": 0,
    "mobil_in": 0,
    "mobil_out": 0,
    "motor_in": 0,
    "motor_out": 0,
    "truk_in": 0,
    "truk_out": 0,
    "bus_in": 0,
    "bus_out": 0,
    "masuk": 0,
    "keluar": 0,
}

log_data = []

c = counts

def load_model(model_path):
    session = ort.InferenceSession(model_path,
        providers=['CUDAExecutionProvider','CPUExecutionProvider']
    )

    tracker = BoTSORTTracker()
    return session, tracker

def open_video(path: str):
    cap = cv2.VideoCapture(path)
    return cap

# PREPROCESS FUNCTION
def preprocess(frame):

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE))

    image = image.astype(np.float32) / 255.0
    image = (image - mean) / std

    image = np.transpose(image, (2, 0, 1))
    image = np.expand_dims(image, axis=0)

    return image.astype(np.float32)

# SOFTMAX
def softmax(x):

    x = x - np.max(x, axis=-1, keepdims=True)

    exp_x = np.exp(x)

    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# FUNCTION
def point_side(point, line_start, line_end, eps=5):

    x, y = point

    x1, y1 = line_start
    x2, y2 = line_end

    value = (
        ((x - x1) * (y2 - y1))
        -
        ((y - y1) * (x2 - x1))
    )

    # dekat garis dianggap 0
    if abs(value) < eps:

        return 0

    return value

def is_crossing_line(prev_point, curr_point, line):

    line_start = line[0]
    line_end = line[1]

    prev_side = point_side(prev_point, line_start, line_end)

    curr_side = point_side(curr_point, line_start, line_end)

    # crossing normal
    if prev_side * curr_side < 0:
        return True

    # salah satu tepat di garis
    if prev_side == 0 and curr_side != 0:
        return True

    if curr_side == 0 and prev_side != 0:
        return True

    return False

def xywh_to_xyxy(box, orig_w, orig_h):
    cx, cy, w, h = box

    x1 = int((cx - w / 2) * orig_w)
    y1 = int((cy - h / 2) * orig_h)
    x2 = int((cx + w / 2) * orig_w)
    y2 = int((cy + h / 2) * orig_h)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(orig_w, x2)
    y2 = min(orig_h, y2)

    return [x1, y1, x2, y2]

def draw_donut_chart(panel, c):

    panel[:] = (35, 35, 35)

    values = [
        c["mobil_in"] + c["mobil_out"],
        c["motor_in"] + c["motor_out"],
        c["truk_in"] + c["truk_out"],
        c["bus_in"] + c["bus_out"]
    ]

    labels = [
        "Mobil",
        "Motor",
        "Truk",
        "Bus"
    ]

    colors = [
        (255, 100, 100),
        (100, 255, 100),
        (100, 100, 255),
        (255, 255, 100)
    ]

    total = sum(values)

    if total == 0:
        total = 1

    center = (150, 110)
    radius = 80

    start_angle = 0

    for value, color in zip(values, colors):

        angle = int((value / total) * 360)

        cv2.ellipse(
            panel,
            center,
            (radius, radius),
            0,
            start_angle,
            start_angle + angle,
            color,
            -1
        )

        start_angle += angle

    cv2.circle(
        panel,
        center,
        40,
        (35, 35, 35),
        -1
    )

    cv2.putText(
        panel,
        "Vehicle Composition",
        (15, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255,255,255),
        2
    )

    y = 40

    for label, value, color in zip(labels, values, colors):

        cv2.rectangle(
            panel,
            (240, y),
            (255, y + 15),
            color,
            -1
        )

        cv2.putText(
            panel,
            f"{label}: {value}",
            (265, y + 13),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255,255,255),
            1
        )

        y += 30

    return panel

def draw_bar_chart(panel, c):

    panel[:] = (35,35,35)

    masuk = c["masuk"]
    keluar = c["keluar"]

    max_value = max(
        masuk,
        keluar,
        1
    )

    chart_bottom = 180

    masuk_height = int(
        (masuk / max_value) * 120
    )

    keluar_height = int(
        (keluar / max_value) * 120
    )

    cv2.putText(
        panel,
        "Traffic Flow",
        (15,25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255,255,255),
        2
    )

    cv2.rectangle(
        panel,
        (80, chart_bottom - masuk_height),
        (140, chart_bottom),
        (0,255,0),
        -1
    )

    cv2.rectangle(
        panel,
        (220, chart_bottom - keluar_height),
        (280, chart_bottom),
        (0,0,255),
        -1
    )

    cv2.putText(
        panel,
        str(masuk),
        (90, chart_bottom - masuk_height - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255,255,255),
        1
    )

    cv2.putText(
        panel,
        str(keluar),
        (230, chart_bottom - keluar_height - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255,255,255),
        1
    )

    cv2.putText(
        panel,
        "IN",
        (90,205),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255,255,255),
        2
    )

    cv2.putText(
        panel,
        "OUT",
        (220,205),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255,255,255),
        2
    )

    return panel

def draw_card(
    img,
    x,
    y,
    w,
    h,
    title,
    value
):

    cv2.rectangle(
        img,
        (x,y),
        (x+w,y+h),
        (45,45,45),
        -1
    )

    cv2.rectangle(
        img,
        (x,y),
        (x+w,y+h),
        (90,90,90),
        2
    )

    cv2.putText(
        img,
        title,
        (x+15,y+35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,255,255),
        2
    )

    cv2.putText(
        img,
        str(value),
        (x+15,y+85),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0,255,255),
        3
    )

def vehicle_counter(model_path, video_path):
    line_in = [
        [393,680],
        [1670,680]
    ]

    line_out = [
        [238,790],
        [1874,790]
    ]

    session, tracker = load_model(model_path)

    input_name = session.get_inputs()[0].name

    # =========================
    # VIDEO SOURCE
    # =========================
    cap = open_video(video_path)

    # Output Video
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = 1280
    height = 720
    output_video = cv2.VideoWriter(video_path.replace(".mp4", "_output.mp4"), fourcc, fps, (width, height))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # =========================
    # RANDOM COLORS
    # =========================
    class_names = COCO_CLASSES
    np.random.seed(42)
    colors = {}

    for i, class_name in enumerate(class_names):
        colors[i] = (
            int(np.random.randint(0, 255)),
            int(np.random.randint(0, 255)),
            int(np.random.randint(0, 255))
        )

    vehicle_class_ids = [3, 4, 6, 8] # car, motorcycle, bus, truck

    # COCO class ID untuk kendaraan
    VEHICLE_CLASS_MAP = {
        3: "mobil",    # car
        4: "motor",    # motorcycle
        6: "bus",      # bus
        8: "truk",     # truck
    }

    track_history = {}

    # menyimpan urutan line yang dilewati
    track_line_history = {}

    # menyimpan object yang sudah dihitung
    counted_in = set()
    counted_out = set()

    # total counter
    total_in = 0
    total_out = 0

    # Loop Video
    frame_count = 0

    donut_panel = np.zeros(
        (220, 500, 3),
        dtype=np.uint8
    )

    bar_panel = np.zeros(
        (220, 300, 3),
        dtype=np.uint8
    )

    update_chart = True
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        orig_h, orig_w = frame.shape[:2]
        frame_count += 1

        # =========================
        # DRAW COUNTING LINES
        # =========================
        cv2.line(frame, tuple(line_in[0]), tuple(line_in[1]), (0, 255, 0), 3)
        cv2.line(frame, tuple(line_out[0]), tuple(line_out[1]), (255, 0, 0),3)

        # =========================
        # PREPROCESS
        # =========================
        input_tensor = preprocess(frame)

        # =========================
        # INFERENCE
        # =========================
        t0 = time.perf_counter()
        outputs = session.run(
            None,
            {input_name: input_tensor}
        )
        infer_ms = (time.perf_counter() - t0) * 1000

        # =========================
        # OUTPUT PARSING
        # =========================
        boxes, labels = outputs

        # =========================
        # POSTPROCESS
        # =========================
        probs = softmax(labels)

        xyxy_list = []
        confidence_list = []
        class_id_list = []

        for i in range(boxes.shape[1]):
            scores = probs[0, i]
            class_id = np.argmax(scores)

            score = scores[class_id]

            if class_id not in vehicle_class_ids:
                continue

            # =========================
            # BOX CONVERSION
            # =========================
            xyxy_list.append(xywh_to_xyxy(boxes[0, i], orig_w, orig_h))
            confidence_list.append(score)
            class_id_list.append(class_id)

        if len(xyxy_list) > 0:
            detections = sv.Detections(
                xyxy=np.array(xyxy_list),
                confidence=np.array(confidence_list),
                class_id=np.array(class_id_list)
            )

            tracked_detections = tracker.update(
                detections
            )

        # =====================================================
        # LOOP TRACKED OBJECTS
        # =====================================================
        for (
            xyxy,
            mask,
            confidence,
            class_id,
            tracker_id,
            data
        ) in tracked_detections:

            if tracker_id == -1:
                continue

            # =================================================
            # BBOX
            # =================================================
            x1, y1, x2, y2 = xyxy.astype(int)

            # =================================================
            # CENTER
            # =================================================
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            current_center = (cx, cy)

            # =================================================
            # INIT TRACK HISTORY
            # =================================================
            if tracker_id not in track_history:

                track_history[tracker_id] = []

            # simpan trajectory
            track_history[tracker_id].append(
                current_center
            )

            # =================================================
            # DRAW CENTER
            # =================================================
            cv2.circle(frame, current_center, 5, (0, 0, 255), -1)

            # =================================================
            # DRAW BBOX
            # =================================================
            class_name = COCO_CLASSES[class_id]
            color = colors.get(class_id, (0, 255, 0))

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # =================================================
            # DRAW TRACK ID
            # =================================================
            label = f"{tracker_id}: {class_name}: {confidence:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            # =================================================
            # MINIMAL 2 POINTS
            # =================================================
            if len(track_history[tracker_id]) < 2:
                continue

            # =================================================
            # PREVIOUS & CURRENT CENTER
            # =================================================
            prev_center = track_history[tracker_id][-2]
            curr_center = track_history[tracker_id][-1]

            # =================================================
            # CHECK CROSSING
            # =================================================
            cross_line_in = is_crossing_line(prev_center, curr_center,line_in)

            cross_line_out = is_crossing_line(prev_center, curr_center,line_out)

            # =================================================
            # INIT LINE HISTORY
            # =================================================
            if tracker_id not in track_line_history:
                track_line_history[tracker_id] = []

            # =================================================
            # SAVE LINE SEQUENCE
            # =================================================

            # crossing line_in
            if cross_line_in:
                if (len(track_line_history[tracker_id]) == 0
                    or
                    track_line_history[tracker_id][-1] != "line_in"):

                    track_line_history[tracker_id].append("line_in")

            # crossing line_out
            if cross_line_out:
                if (len(track_line_history[tracker_id]) == 0
                    or
                    track_line_history[tracker_id][-1] != "line_out"):

                    track_line_history[tracker_id].append("line_out")

            # =================================================
            # GET HISTORY
            # =================================================
            history = track_line_history[tracker_id]

            # =================================================
            # COUNT IN
            # =================================================
            #
            # line_in -> line_out
            #
            if (
                history == ["line_in", "line_out"]
                and
                tracker_id not in counted_in
            ):

                total_in += 1

                counted_in.add(tracker_id)
                # Tentukan kategori kendaraan
                category = VEHICLE_CLASS_MAP.get(class_id, None)

                # Update session_state
                c["masuk"] += 1
                c["total"] += 1

                update_chart = True

                log_data.insert(0, {
                    "ts": datetime.now(),
                    "veh": category,
                    "status": "Masuk"
                })

                if category:
                    c[f"{category}_in"] += 1

            # =================================================
            # COUNT OUT
            # =================================================
            #
            # line_out -> line_in
            #
            elif (
                history == ["line_out", "line_in"]
                and
                tracker_id not in counted_out
            ):

                total_out += 1

                counted_out.add(tracker_id)
                category = VEHICLE_CLASS_MAP.get(class_id, None)

                # Update session_state
                c["keluar"] += 1
                c["total"] += 1

                update_chart = True

                log_data.insert(0, {
                    "ts": datetime.now(),
                    "veh": category,
                    "status": "Keluar"
                })

                if category:
                    c[f"{category}_out"] += 1

        # FPS overlay
        cv2.putText(frame, f"Infer: {infer_ms:.1f}ms", (10, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 128), 2, cv2.LINE_AA)

        dashboard = np.zeros(
            (720,1280,3),
            dtype=np.uint8
        )

        dashboard[:] = (20,20,20)

        video_panel = cv2.resize(frame, (800, 450))

        draw_card(dashboard,850,20,180,90,"MASUK",c["masuk"])
        draw_card(dashboard,1050,20,180,90,"KELUAR",c["keluar"])

        draw_card(dashboard,850,220,180,90,"MOBIL",c["mobil_in"] + c["mobil_out"])
        draw_card(dashboard,1050,220,180,90,"MOTOR",c["motor_in"] + c["motor_out"])

        draw_card(dashboard,850,320,180,90,"TRUK",c["truk_in"] + c["truk_out"])
        draw_card(dashboard,1050,320,180,90,"BUS",c["bus_in"] + c["bus_out"])

        if update_chart:

            donut_panel = draw_donut_chart(donut_panel, c)

            bar_panel = draw_bar_chart(bar_panel, c)

            update_chart = False

        dashboard[20:470, 20:820] = video_panel

        dashboard[480:700, 20:520] = donut_panel

        dashboard[480:700, 540:840] = bar_panel

        # Log

        cv2.rectangle(
            dashboard,
            (860, 480),
            (1260, 700),
            (40,40,40),
            -1
        )

        cv2.putText(dashboard,"LOG",(880, 510),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

        y = 540

        for row in log_data[:6]:

            text = (
                f"{row['veh']} - "
                f"{row['ts'].strftime('%H:%M:%S')} - "
                f"{row['status']}"
            )

            cv2.putText(
                dashboard,
                row["ts"].strftime("%H:%M:%S"),
                (880, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (255,255,255),
                1
            )

            cv2.putText(
                dashboard,
                row["veh"],
                (980, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (255,255,255),
                1
            )

            cv2.putText(
                dashboard,
                row["status"],
                (1080, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (255,255,255),
                1
            )

            y += 25

        output_video.write(dashboard)
        print("Frame Processed", frame_count, " : ", total_frames)

    cap.release()
    output_video.release()
    cv2.destroyAllWindows()

In [ ]:
vehicle_counter("rfdetr-nano.onnx", "inference/highway.mp4")

In [ ]:
from google.colab import files

files.download("inference/highway_output.mp4")